# House Prices ML Project

This notebook builds a machine-learning model to predict house sale prices using the Kaggle House Prices dataset.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Load the Dataset

The uploaded training file is named `train1.csv`.

In [ ]:
df = pd.read_csv('dataset/train1.csv')
print('Dataset shape:', df.shape)
df.head()

## 3. Understand the Dataset

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
df.describe(include='all').T.head(20)

## 4. Separate Features and Target

`SalePrice` is the target variable we want to predict.

In [ ]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

print('Features:', X.shape)
print('Target:', y.shape)

## 5. Identify Numerical and Categorical Columns

In [ ]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numerical columns:', len(numeric_features))
print('Categorical columns:', len(categorical_features))

## 6. Preprocess the Data

Missing numerical values are filled with the median. Missing categorical values are filled with the most frequent value, and categorical columns are converted using one-hot encoding.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 7. Split the Dataset

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training rows:', len(X_train))
print('Validation rows:', len(X_valid))

## 8. Train a Random Forest Model

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_valid)

## 9. Evaluate the Model

In [ ]:
rmse = np.sqrt(mean_squared_error(y_valid, rf_predictions))
mae = mean_absolute_error(y_valid, rf_predictions)
r2 = r2_score(y_valid, rf_predictions)

print(f'RMSE: {rmse:,.2f}')
print(f'MAE : {mae:,.2f}')
print(f'R²  : {r2:.4f}')

## 10. Actual vs Predicted Prices

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_valid, rf_predictions, alpha=0.6)
plt.xlabel('Actual Sale Price')
plt.ylabel('Predicted Sale Price')
plt.title('Actual vs Predicted House Prices')
plt.show()

## 11. Train the Final Model

After evaluating the model, train it on the complete training dataset.

In [ ]:
rf_model.fit(X, y)
print('Final model trained successfully.')

## 12. Optional Kaggle Test Prediction

If `test.csv` is later placed inside the `dataset` folder, run this cell to create `submission.csv`.

In [ ]:
from pathlib import Path

test_path = Path('dataset/test.csv')

if test_path.exists():
    test_df = pd.read_csv(test_path)
    test_predictions = rf_model.predict(test_df)
    submission = pd.DataFrame({
        'Id': test_df['Id'],
        'SalePrice': test_predictions
    })
    submission.to_csv('submission.csv', index=False)
    print('submission.csv created successfully.')
    display(submission.head())
else:
    print('dataset/test.csv was not found. Add test.csv to the dataset folder when available.')

## Conclusion

The project loads the House Prices dataset, handles missing values and categorical variables, trains a Random Forest regression model, evaluates its performance, and provides an optional Kaggle submission workflow.